In [ ]:
# Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Optional, List

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

## 1. Load Sample Data

In [ ]:
# Load datasets
df_sales = pd.read_csv("../data/demo_sales.csv")
df_titanic = pd.read_csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")

print(f"✅ Sales: {df_sales.shape}")
print(f"✅ Titanic: {df_titanic.shape}")

## 2. Visualization Helper Class

In [ ]:
class DataVisualizer:
    """
    A simple visualization helper for data analysis.
    Provides common plots for EDA.
    """
    
    def __init__(self, df: pd.DataFrame, figsize: tuple = (10, 6)):
        self.df = df
        self.figsize = figsize
    
    def plot_distributions(self, columns: Optional[List[str]] = None, bins: int = 30):
        """Plot distributions for numeric columns."""
        if columns is None:
            columns = self.df.select_dtypes(include=[np.number]).columns.tolist()
        
        n_cols = min(3, len(columns))
        n_rows = (len(columns) + n_cols - 1) // n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
        axes = np.array(axes).flatten() if len(columns) > 1 else [axes]
        
        for i, col in enumerate(columns):
            ax = axes[i]
            self.df[col].hist(bins=bins, ax=ax, edgecolor='black', alpha=0.7)
            ax.set_title(f'{col}', fontsize=12)
            ax.set_xlabel(col)
            ax.set_ylabel('Frequency')
        
        # Hide empty subplots
        for j in range(i+1, len(axes)):
            axes[j].set_visible(False)
        
        plt.tight_layout()
        plt.suptitle('Distribution of Numeric Features', y=1.02, fontsize=14)
        plt.show()
    
    def plot_correlation_heatmap(self):
        """Plot correlation heatmap for numeric columns."""
        numeric_df = self.df.select_dtypes(include=[np.number])
        
        if len(numeric_df.columns) < 2:
            print("Not enough numeric columns for correlation")
            return
        
        corr = numeric_df.corr()
        
        plt.figure(figsize=self.figsize)
        mask = np.triu(np.ones_like(corr, dtype=bool))
        sns.heatmap(corr, mask=mask, annot=True, cmap='RdBu_r', center=0,
                    fmt='.2f', square=True, linewidths=0.5)
        plt.title('Correlation Heatmap', fontsize=14)
        plt.tight_layout()
        plt.show()
    
    def plot_categorical_counts(self, columns: Optional[List[str]] = None, top_n: int = 10):
        """Plot value counts for categorical columns."""
        if columns is None:
            columns = self.df.select_dtypes(include=['object', 'category']).columns.tolist()
        
        if not columns:
            print("No categorical columns found")
            return
        
        n_cols = min(2, len(columns))
        n_rows = (len(columns) + n_cols - 1) // n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 4*n_rows))
        axes = np.array(axes).flatten() if len(columns) > 1 else [axes]
        
        for i, col in enumerate(columns):
            ax = axes[i]
            counts = self.df[col].value_counts().head(top_n)
            counts.plot(kind='barh', ax=ax, color=sns.color_palette('husl', len(counts)))
            ax.set_title(f'{col}', fontsize=12)
            ax.set_xlabel('Count')
        
        for j in range(i+1, len(axes)):
            axes[j].set_visible(False)
        
        plt.tight_layout()
        plt.suptitle('Categorical Feature Distributions', y=1.02, fontsize=14)
        plt.show()
    
    def plot_target_analysis(self, target: str, features: Optional[List[str]] = None):
        """Analyze relationship between features and target."""
        if target not in self.df.columns:
            print(f"Target '{target}' not found")
            return
        
        # Determine if target is numeric or categorical
        is_numeric_target = pd.api.types.is_numeric_dtype(self.df[target])
        
        if features is None:
            if is_numeric_target:
                features = self.df.select_dtypes(include=[np.number]).columns.tolist()
            else:
                features = self.df.columns.tolist()
            features = [f for f in features if f != target][:6]
        
        n_cols = min(3, len(features))
        n_rows = (len(features) + n_cols - 1) // n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
        axes = np.array(axes).flatten() if len(features) > 1 else [axes]
        
        for i, feat in enumerate(features):
            ax = axes[i]
            is_numeric_feat = pd.api.types.is_numeric_dtype(self.df[feat])
            
            if is_numeric_target and is_numeric_feat:
                ax.scatter(self.df[feat], self.df[target], alpha=0.5)
                ax.set_xlabel(feat)
                ax.set_ylabel(target)
            elif not is_numeric_target and is_numeric_feat:
                self.df.boxplot(column=feat, by=target, ax=ax)
                ax.set_title(feat)
            else:
                ct = pd.crosstab(self.df[feat], self.df[target])
                ct.plot(kind='bar', ax=ax, stacked=True)
                ax.set_xlabel(feat)
            
            ax.set_title(f'{feat} vs {target}', fontsize=10)
        
        for j in range(i+1, len(axes)):
            axes[j].set_visible(False)
        
        plt.tight_layout()
        plt.suptitle(f'Target Analysis: {target}', y=1.02, fontsize=14)
        plt.show()
    
    def plot_missing_values(self):
        """Visualize missing values."""
        missing = self.df.isnull().sum()
        missing = missing[missing > 0].sort_values(ascending=True)
        
        if len(missing) == 0:
            print("✅ No missing values!")
            return
        
        plt.figure(figsize=(10, max(4, len(missing) * 0.3)))
        missing.plot(kind='barh', color='coral')
        plt.xlabel('Missing Count')
        plt.title('Missing Values by Column')
        plt.tight_layout()
        plt.show()

## 3. Visualize Sales Data (Regression)

In [ ]:
viz_sales = DataVisualizer(df_sales)

# Distribution plots
viz_sales.plot_distributions()

In [ ]:
# Correlation heatmap
viz_sales.plot_correlation_heatmap()

In [ ]:
# Categorical analysis
viz_sales.plot_categorical_counts()

## 4. Visualize Titanic Data (Classification)

In [ ]:
viz_titanic = DataVisualizer(df_titanic)

# Missing values
viz_titanic.plot_missing_values()

In [ ]:
# Numeric distributions
viz_titanic.plot_distributions(['Age', 'Fare', 'Pclass'])

In [ ]:
# Correlation
viz_titanic.plot_correlation_heatmap()

In [ ]:
# Target analysis
viz_titanic.plot_target_analysis('Survived', ['Pclass', 'Sex', 'Age', 'Fare'])

## ✅ Summary

This module provides:
- `DataVisualizer` class with methods:
  - `plot_distributions()` - Histograms for numeric columns
  - `plot_correlation_heatmap()` - Correlation matrix visualization
  - `plot_categorical_counts()` - Bar charts for categorical columns
  - `plot_target_analysis()` - Feature vs target relationships
  - `plot_missing_values()` - Missing value visualization